Github repository is here: https://github.com/ong8181/DASC3240

# Visualization of high dimensional data: PCA and others

## Reference and datasets
- `sklearn.decomposition.PCA`: https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html#examples-using-sklearn-decomposition-pca
- `sklearn.metrics.pairwise_distances`: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise_distances.html
- `sklearn.manifold.MDS`: https://scikit-learn.org/stable/modules/generated/sklearn.manifold.MDS.html
- `sklearn.manifold.TSNE`: https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html
- lambda function (or anonymous function): A simple method to define a simple fuction `lambda parameters: expression` (https://docs.python.org/3/reference/expressions.html#lambda)

# Contents
1. Principal Component Analysis (PCA) with `iris`
2. Principal Component Analysis (PCA) with higher dimensional data
3. NMDS, PCoA and t-SNE

# 1. Principal Component Analysis (PCA) with `iris`
First, we will perform PCA with a relatively simple data, `iris` (data with only four dimensions).


In [ ]:
# Load libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load data
d_iris = pd.read_csv("data/iris.csv", usecols=range(1,6))
d_iris.head()

# A method to load iris from sklearn.datasets package
#from sklearn.datasets import load_iris
#d = load_iris()

Then, let's load the function `PCA` to do the analysis.

In [ ]:
# Load library for PCA and other machine learning tools
import sklearn
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler    # Normalization
from sklearn.preprocessing import StandardScaler  # Standardization

First, we will check the overall data structure.

In [ ]:
# Import Seaborn
import seaborn as sns
sns.pairplot(data = d_iris, hue = "Species", palette = "colorblind");

- Data standardization and normalizatoin
  - Standardization  
    $x_{std} = \frac{x - \mu}{\sigma}$
  - Normalization  
    $x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$

In [ ]:
# Standardize the data
ds_iris = pd.DataFrame(StandardScaler().fit_transform(d_iris.iloc[:, 0:4])) # Standard Deviation is used (not unbiased standard deviation) 
#ds_iris = d_iris.iloc[:, 0:4].apply(lambda x: (x-x.mean())/x.std(), axis=0) # Standardization

# Or Normalize the data
#ds_iris = pd.DataFrame(MinMaxScaler().fit_transform(d_iris.iloc[:, 0:4]))
#ds_iris = d_iris.iloc[:, 0:4].apply(lambda x: (x-x.min())/(x.max()-x.min()), axis=0) # Normalization

# Check data
ds_iris.head()

In [ ]:
# Do PCA
pca_iris = PCA()
pca_ds_iris = pca_iris.fit_transform(ds_iris) # Transform data
pca_ds_iris.shape

## Alternative code
#pca_ds_iris = PCA(n_components=4).fit_transform(ds_iris)
#pca_ds_iris.shape

In [ ]:
# Check content
pca_df_iris = pd.DataFrame(pca_ds_iris, columns=["PC1", "PC2", "PC3", "PC4"])
pca_df_iris.head

In [ ]:
# Add species information
pca_df_iris2 = pd.concat([pca_df_iris, d_iris], axis = 1)

# Plot the PCA result
sns.scatterplot(x = "PC1", y = "PC2", data = pca_df_iris2, hue = "Species")
plt.legend(loc="upper left", bbox_to_anchor=(1,1));

## Check PCA results
Some important results of PCA are:
- The propotion of variation explained (how much information was lost?)
- Eigenvector (which factor contributed to PCA1, PCA2, ... ?)

In [ ]:
## Check PCA results
print(pca_iris.explained_variance_ratio_)

This means that PC1 (the first component, or axis), PC2, PC3, and PC4 explained 72.96%, 22.85%, 3.67%, and 0.52% of the total variation.

In [ ]:
## Sum of variation explained (should be 100%)
sum(pca_iris.explained_variance_ratio_)

In [ ]:
print(pca_iris.components_)

Interpretation is that "Sepal.Length" contributed `0.52106591`, `0.37741762`, `-0.71956635`, and `-0.26128628` to PC1, PC2, PC3, and PC4, respectively. "Sepal.Width" contributed `-0.26934744`, `0.92329566`, `0.24438178`, and `0.12350962` to PC1, PC2, PC3, and PC4, respectively.  

Documentations about `sklearn.decomposition.PCA()` can be found [here](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html#examples-using-sklearn-decomposition-pca).

# 2. Principal Component Analysis (PCA) with higher dimensional data
Then, let's use larger data and do PCA. As for high dimensional data, let's use fish monitoring data from Ushio et al. (2023) _eLife_.

## Brief background of the data
In Ushio et al. (2023) _eLife_, water samples were collected twice a month from August 2017 to August 2019 in Chiba Prefecture, Japan. Then, DNA was extracted from water samples, and fish DNA in the samples were analyzed (fish DNA can be detected from water samples!! [please recall this video](https://www.youtube.com/watch?v=TQdTV1rAlWY)). We have made all data public at [Github](https://github.com/ong8181/eDNA-BosoFish-network) (MIT license; anyone can reanalyze it). The graphical abstract of the study is below:  
<img src="img/L13_Ushio2023eLife_graphycal_abst.png" alt="" width="700">

I edited the data, and you can first use [this CSV file](https://github.com/ong8181/DASC3240/blob/main/data/eDNA_copy_site_top.csv). In the CSV file, the first three columns indicate the study sites where we collected water samples (i.e., site name, latitude, and longitude). The other columns indicate IDs of 50 most dominant fish species, and each value indicate fish DNA copies / ml seawater (copies = an index of fish DNA concentration). If you are interested in fish taxa and study site information, you can check the [taxa data](https://github.com/ong8181/eDNA-BosoFish-network/blob/main/data/tax_sheet.csv) and [study site data](https://github.com/ong8181/eDNA-BosoFish-network/blob/main/data/sample_sheet.csv) in [another Github repository](https://github.com/ong8181/eDNA-BosoFish-network/).

First, we will laod the data to Python.

In [ ]:
# Load data
d = pd.read_csv("data/eDNA_copy_site_top.csv", index_col = 0)
## From URL
#d = pd.read_csv("https://github.com/ong8181/DASC3240/blob/main/data/eDNA_copy_site_top.csv", index_col = 0)
d.head()

## Do fish community compositions cluster depending on the study sites?
Next, we will briefly check whether the fish community compositions depend on the study sites. PCA (or other dimension reduction techniques) is useful to (intuitively) answer this question.

In [ ]:
# Method 1: Not standardize the data
ds = d.iloc[:, 3:53]

# Method 2: Standardize the data
#ds = pd.DataFrame(StandardScaler().fit_transform(d.iloc[:, 3:53])) # Standard Deviation is used 
#ds = d.iloc[:, 3:53].apply(lambda x: (x-x.mean())/x.std(), axis=0)

# Method 3: If you are interested in the compositional data
# Convert the data to "compositional data" (relative abundance, %, data)
#d_fish = d.iloc[:, 3:53]
#ds = d_fish.div(d_fish.sum(axis = 0), axis = 1)

# Do PCA
pca = PCA()
pca_ds = pca.fit_transform(ds)

# Check content
pca_df = pd.DataFrame(pca_ds)
pca_df.head

### Visualize the pattern
Then, let's plot only the first two PC axes.

In [ ]:
# Rename column names (preparation)
new_columns = {i: f'PC{i+1}' for i in range(len(pca_df.columns))}

# Rename column names
pca_df = pca_df.rename(columns = new_columns)
pca_df.head

In [ ]:
# Add site information
pca_df2 = pd.concat([pca_df, d["site_name"].reset_index()], axis = 1)

# Plot the PCA result
sns.scatterplot(x = "PC1", y = "PC2", data = pca_df2, hue = "site_name")
plt.legend(loc="upper left", bbox_to_anchor=(1,1));

In [ ]:
# Check PCA results
print(pca.explained_variance_ratio_)

# 3. PCoA, NMDS and t-SNE
Lastly, we can look at how NMDS, PCoA, and t-SNE can be implemented using the same fish data.

In [ ]:
# Load libraries
#import pandas as pd
#import numpy as np
#import matplotlib.pyplot as plt
#from sklearn.decomposition import PCA
from sklearn.manifold import MDS, TSNE
from sklearn.metrics import pairwise_distances

### PCoA (or MDS)

In [ ]:
# Calculate the distance matrix (here we use "manhattan" distance)
distance_matrix1 = pairwise_distances(ds, metric = "manhattan")

# Perform PCoA
mds = MDS(n_components = 2, dissimilarity = "precomputed")
mds_result = mds.fit_transform(distance_matrix1)

# Store the results as pd.DataFrame
mds_df = pd.DataFrame(mds_result, columns=["Axis1", "Axis2"])
mds_df.head

In [ ]:
# Add site information
mds_df2 = pd.concat([mds_df, d["site_name"].reset_index()], axis = 1)

# Plot the PCA result
sns.scatterplot(x = "Axis1", y = "Axis2", data = mds_df2, hue = "site_name")
plt.legend(loc="upper left", bbox_to_anchor=(1,1));

### NMDS

In [ ]:
# Seed for consistency
np.random.seed(1234)

# Calculate the distance matrix (here we use "manhattan" distance)
distance_matrix2 = pairwise_distances(ds, metric = "braycurtis")

# Perform PCoA
nmds = MDS(n_components = 2, dissimilarity = "precomputed", metric = False)
nmds_result = nmds.fit_transform(distance_matrix2)

# Store the results as pd.DataFrame
nmds_df = pd.DataFrame(nmds_result, columns=["Axis1", "Axis2"])
nmds_df.head

In [ ]:
# Add site information
nmds_df2 = pd.concat([nmds_df, d["site_name"].reset_index()], axis = 1)

# Plot the PCA result
sns.scatterplot(x = "Axis1", y = "Axis2", data = nmds_df2, hue = "site_name")
plt.legend(loc="upper left", bbox_to_anchor=(1,1));

### t-SNE

In [ ]:
#import pandas as pd
#import matplotlib.pyplot as plt
#from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

In [ ]:
# Seed for consistency
np.random.seed(1234)

# Perform t-SNE
tsne = TSNE(n_components = 2)
tsne_result = tsne.fit_transform(ds)

# Store the results as pd.DataFrame
tsne_df = pd.DataFrame(tsne_result, columns=["Axis1", "Axis2"])
tsne_df.head

In [ ]:
# Add site information
tsne_df2 = pd.concat([tsne_df, d["site_name"].reset_index()], axis = 1)

# Plot the PCA result
sns.scatterplot(x = "Axis1", y = "Axis2", data = tsne_df2, hue = "site_name")
plt.legend(loc="upper left", bbox_to_anchor=(1,1))
# Save the figure
plt.savefig("img/L13_Figure_tSNE_woCap.png", format = "png", bbox_inches = "tight")
plt.show()

### Challanges
- Change the random seed for NMDS and t-SNA, and then check how the results change.

# Additional topic: Figure caption


### Matplotlib
Here, let's reuse the previous figure and add a figure caption.

In [ ]:
# Plot the PCA result
sns.scatterplot(x = "Axis1", y = "Axis2", data = tsne_df2, hue = "site_name")

## Add legend
plt.legend(loc = "upper left", bbox_to_anchor = (1,1))

## Add caption
cap = "Figure 1 | t-SNE plot of the dominant fish community composition based on eDNA survey. Axis1 and Axis2 represent t-SNE axses. Each point represents each eDNA sample, and colors indicate study sites."
plt.figtext(0, -0.15, cap, wrap = True, horizontalalignment = "left", fontsize = 12)

# Save the figure
plt.savefig("img/L13_Figure_tSNE_wCap.png", format = "png", bbox_inches = "tight")

plt.show()

## HTML and Markdown examples
The next example is using Markdown.

<figure>
<img src="img/L13_Figure_tSNE_woCap.png" style="width:80%">
<figcaption align = "left">Figure 1 | t-SNE plot of the dominant fish community composition based on eDNA survey. Axis1 and Axis2 represent t-SNE axses. Each point represents each eDNA sample, and colors indicate study sites.</figcaption>
</figure>

![Fig1](img/L13_Figure_tSNE_woCap.png)  
**Figure 1**| t-SNE plot of the dominant fish community composition based on eDNA survey. Axis1 and Axis2 represent t-SNE axses. Each point represents each eDNA sample, and colors indicate study sites.